# 04 — Quiver access diagnostic & House validation 2024

## Rappel critique

Quiver est une validation externe. House reste la source officielle.

Aucune correction automatique de House via Quiver.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.quiver_client import (
    get_quiver_token, diagnose_congresstrading_access, fetch_congresstrading_pages,
    normalize_quiver_records, filter_house_year, compare_house_quiver_coverage,
    write_quiver_report,
)
from src.utils import AUDIT_DIR, EXTERNAL_QUIVER_DIR, PROCESSED_HOUSE_DIR, require_file

print("Imports OK")

In [ ]:
YEAR = 2024
PAGE_SIZE = 500
MAX_PAGES = 3  # Augmenter seulement après diagnostic.

if load_dotenv is not None:
    load_dotenv(ROOT / ".env")

token = get_quiver_token()
print("QUIVER_API_TOKEN present:", bool(token))

In [ ]:
diagnostic_path = AUDIT_DIR / "quiver_api_access_diagnostic.json"
diagnostic = diagnose_congresstrading_access(token=token, output_path=diagnostic_path)

print("diagnostic status:", diagnostic.get("status"))
print("accessible version:", diagnostic.get("accessible_version"))
print("Written:", diagnostic_path.relative_to(ROOT))

In [ ]:
records = []
version = diagnostic.get("accessible_version")

if token and version:
    raw_path = EXTERNAL_QUIVER_DIR / "quiver_congress_trading_raw_sample.json"
    records = fetch_congresstrading_pages(
        token=token,
        version=version,
        page_size=PAGE_SIZE,
        max_pages=MAX_PAGES,
        output_raw_path=raw_path,
    )
    print("records fetched:", len(records))
else:
    print("Skipped: no Quiver token or no accessible version.")

In [ ]:
if records:
    quiver_df = normalize_quiver_records(records, schema_version=version)
else:
    quiver_df = pd.DataFrame()

print("quiver_df shape:", quiver_df.shape)
if not quiver_df.empty:
    display(quiver_df.head())
    display(quiver_df.columns.to_series())

In [ ]:
quiver_house = filter_house_year(quiver_df, YEAR) if not quiver_df.empty else pd.DataFrame()
quiver_2024_path = EXTERNAL_QUIVER_DIR / f"quiver_congress_trading_{YEAR}.csv"

if not quiver_house.empty:
    quiver_house.to_csv(quiver_2024_path, index=False)
    print("Written:", quiver_2024_path.relative_to(ROOT))
else:
    print("No Quiver House records for selected year in fetched pages.")

In [ ]:
house_2024_path = PROCESSED_HOUSE_DIR / f"ptr_index_{YEAR}.csv"

if house_2024_path.exists():
    house_2024 = pd.read_csv(house_2024_path, dtype={"doc_id": str})
    print("house_2024 shape:", house_2024.shape)
    display(house_2024.head())
else:
    house_2024 = pd.DataFrame()
    print("Missing:", house_2024_path.relative_to(ROOT))

In [ ]:
metrics = None
validation_path = AUDIT_DIR / f"quiver_house_validation_{YEAR}.csv"

if not house_2024.empty and not quiver_house.empty:
    validation_df, metrics = compare_house_quiver_coverage(
        house_2024,
        quiver_house,
        output_path=validation_path,
    )
    display(validation_df)
    print("Written:", validation_path.relative_to(ROOT))
else:
    print("Coverage comparison skipped.")

In [ ]:
if not house_2024.empty and not quiver_house.empty:
    house_keys = set(house_2024["declarant_name_raw"].dropna().str.lower().str.strip())
    quiver_keys = set(quiver_house["name"].dropna().str.lower().str.strip())
    print("House-only examples:", sorted(list(house_keys - quiver_keys))[:20])
    print("Quiver-only examples:", sorted(list(quiver_keys - house_keys))[:20])
else:
    print("Divergence examples skipped.")

In [ ]:
report_path = write_quiver_report(diagnostic, metrics=metrics)
print("Written:", report_path.relative_to(ROOT))

## Conclusion

**OK** si le diagnostic Quiver est écrit et les limites sont explicites.

**NEXT** — ne pas corriger House avec Quiver ; utiliser les divergences comme audit.